### Phân tách LU Toàn Tập (Kiểm tra điều kiện & Tự động Pivot)\n

In [2]:
import numpy as np
from IPython.display import display, Math, Markdown

def _mat(M, p=4):
    if hasattr(M[0], "__len__"):
        rows = " \\\\ ".join([" & ".join([f"{x:.{p}f}" for x in row]) for row in M])
        return f"\\begin{{bmatrix}} {rows} \\end{{bmatrix}}"
    inner = " \\\\ ".join([f"{x:.{p}f}" for x in M])
    return f"\\begin{{bmatrix}} {inner} \\end{{bmatrix}}^T"

def LU_Decomposition(A_input, b_input=None):
    A = np.array(A_input, dtype=float)
    n = A.shape[0]
    
    display(Markdown("## ❖ PHÂN TÁCH LU (KIỂM TRA VÀ ÁP DỤNG CƠ BẢN / HOÁN VỊ)"))
    display(Math(f"A = {_mat(A)}"))
    
    display(Markdown("### 1. Kiểm tra điều kiện (Các định thức con chính)"))
    minors = []
    can_basic_LU = True
    for k in range(1, n+1):
        sub_A = A[:k, :k]
        det_k = np.linalg.det(sub_A)
        minors.append(det_k)
        display(Math(f"\\Delta_{k} = \\det(A_{{{k}\\times {k}}}) = {det_k:.4f}"))
        if abs(det_k) < 1e-12:
            can_basic_LU = False
            
    if can_basic_LU:
        display(Markdown("=> **Tất cả $\\Delta_k \\neq 0$. Đủ điều kiện phân tách LU cơ bản ($A = LU$).**"))
        # Run Basic LU
        L = np.eye(n)
        U = A.copy()
        for k in range(n - 1):
            display(Markdown(f"#### Khử cột $k = {k+1}$"))
            for i in range(k + 1, n):
                if abs(U[k, k]) < 1e-14: continue
                m = U[i, k] / U[k, k]
                L[i, k] = m
                U[i, :] -= m * U[k, :]
                display(Math(f"l_{{{i+1},{k+1}}} = \\frac{{U_{{{i+1},{k+1}}}}}{{U_{{{k+1},{k+1}}}}} = {m:.4f}"))
            display(Math(f"U \\leftarrow {_mat(U)}"))
            
        display(Markdown("### 2. Kết quả phân tách LU cơ bản"))
        display(Math(f"L = {_mat(L)}"))
        display(Math(f"U = {_mat(U)}"))
        display(Markdown("**Kiểm tra $L \\cdot U = A$:**"))
        display(Math(f"L \\cdot U = {_mat(L @ U)}"))
        P = np.eye(n)
    else:
        display(Markdown("=> **Có định thức con chính bằng 0. KHÔNG đủ điều kiện phân tách LU cơ bản.**\n=> **Sử dụng LU có Hoán vị (Pivoting, $PA = LU$).**"))
        # Run LU Pivoting
        P = np.eye(n)
        L = np.zeros((n, n))
        U = A.copy()
        
        for k in range(n - 1):
            display(Markdown(f"#### Cột $k = {k+1}$"))
            max_row = k + np.argmax(np.abs(U[k:, k]))
            if max_row != k:
                U[[k, max_row], :] = U[[max_row, k], :]
                P[[k, max_row], :] = P[[max_row, k], :]
                if k > 0:
                    L[[k, max_row], :k] = L[[max_row, k], :k]
                display(Markdown(f"**Hoán vị hàng {k+1} và hàng {max_row+1}** (do phần tử $|U_{{{max_row+1},{k+1}}}|$ lớn nhất):"))
                display(Math(f"U \\leftarrow {_mat(U)}"))
                
            if abs(U[k, k]) < 1e-14:
                display(Markdown(f"⚠️ Cảnh báo: Cột {k+1} đã bằng 0, không thể khử (ma trận suy biến)."))
                continue
                
            for i in range(k + 1, n):
                m = U[i, k] / U[k, k]
                L[i, k] = m
                U[i, :] -= m * U[k, :]
                display(Math(f"l_{{{i+1},{k+1}}} = \\frac{{U_{{{i+1},{k+1}}}}}{{U_{{{k+1},{k+1}}}}} = {m:.4f}"))
            display(Math(f"U \\leftarrow {_mat(U)}"))
            L[k, k] = 1.0
        L[n-1, n-1] = 1.0
        
        display(Markdown("### 2. Kết quả phân tách $PA = LU$"))
        display(Math(f"P = {_mat(P)}"))
        display(Math(f"L = {_mat(L)}"))
        display(Math(f"U = {_mat(U)}"))
        display(Markdown("**Kiểm tra $PA = LU$:**"))
        display(Math(f"PA = {_mat(P @ A)}"))
        display(Math(f"LU = {_mat(L @ U)}"))

    if b_input is not None:
        display(Markdown("### 3. Giải hệ phương trình $Ax = b$"))
        b = np.array(b_input, dtype=float).flatten()
        pb = P @ b
        if not can_basic_LU:
            display(Math(f"Pb = {_mat(pb)}"))
            
        display(Markdown("#### Giải $Ly = Pb$ (Thay thế tiến)"))
        y = np.zeros(n)
        for i in range(n):
            y[i] = pb[i] - sum(L[i, j]*y[j] for j in range(i))
            display(Math(f"y_{{{i+1}}} = {y[i]:.6f}"))
            
        display(Markdown("#### Giải $Ux = y$ (Thay thế lùi)"))
        x = np.zeros(n)
        for i in range(n - 1, -1, -1):
            x[i] = (y[i] - sum(U[i, j]*x[j] for j in range(i + 1, n))) / U[i, i]
            display(Math(f"x_{{{i+1}}} = {x[i]:.6f}"))
            
        display(Markdown("**Kiểm tra lại $Ax \\approx b$:**"))
        display(Math(f"Ax = {_mat(A @ x)} \\approx b = {_mat(b)}"))
        return L, U, P, x
    return L, U, P

# ═══════════════════════════════════════════════════════════════════
# NHẬP DỮ LIỆU CỦA BẠN VÀO ĐÂY
# ═══════════════════════════════════════════════════════════════════
# Ma trận ví dụ này CÓ định thức con chính thứ 2 bằng 0 nên cần Pivot
A = np.array([
    [2.0, 4.0,  -2.0],
    [1.0, 2.0,  1.0],
    [4.0, -1.0, 3.0]
], dtype=float)

b = np.array([2.0, 4.0, 6.0], dtype=float) 
# ═══════════════════════════════════════════════════════════════════

_ = LU_Decomposition(A, b)


## ❖ PHÂN TÁCH LU (KIỂM TRA VÀ ÁP DỤNG CƠ BẢN / HOÁN VỊ)

<IPython.core.display.Math object>

### 1. Kiểm tra điều kiện (Các định thức con chính)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

**Kiểm tra lại $Ax \approx b$:**

<IPython.core.display.Math object>